# Mutual Fund Analysis: Data Source Exploration

> **Purpose:** Systematically explore every data source, library, and API idea from `docs/data-source-exploration.md` to understand what data is available, its quality, coverage gaps, and suitability for the Mutual Fund Analysis platform.
>
> **Outcome:** By the end of this notebook we will have a clear picture of:
> - What each library/API provides (fields, frequency, depth)
> - Known limitations for Indian mutual funds
> - Which source is best for each data need (NAV, holdings, benchmarks, risk metrics, etc.)
> - A recommended data architecture for the backend

---

## Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [mftool — AMFI NAV & Scheme Data](#2-mftool)
3. [mfapi.in — JSON NAV API](#3-mfapiin)
4. [mf.captnemo.in — ISIN-based Fund Info](#4-mfcaptnemo)
5. [yfinance — Yahoo Finance (Stocks + Funds)](#5-yfinance)
6. [yahooquery — Yahoo Finance Extended](#6-yahooquery)
7. [mstarpy — Morningstar Data](#7-mstarpy)
8. [jugaad_data / nselib — NSE Benchmark Indices](#8-nse-index-data)
9. [AMFI Direct API — Scheme Universe](#9-amfi-direct)
10. [IndianAPI.in Exploration](#10-indianapiin-exploration)
11. [BSE Mutual Fund Data](#11-bse-mutual-fund-data)
12. [Additional NSE/BSE Libraries](#12-additional-python-libraries-for-nsebse)
13. [Computed Analytics from NAV Data](#13-computed-analytics-from-nav-data)
14. [Forecasting Approaches](#14-forecasting-approaches)
15. [News and Sentiment Research](#15-news--sentiment-research)
16. [Data Coverage Matrix and Recommendations](#16-data-coverage-matrix--recommendations)
17. [Recommended Architecture](#17-recommended-architecture)

---
## 1. Environment Setup

Install all libraries needed for this exploration. Run this cell once, then restart the kernel before executing source checks.

The flow is organized as primary Indian mutual-fund sources, enrichment and benchmark candidates, derived analytics, and implementation recommendations. Network-backed cells may be temporarily unavailable when a provider rate-limits requests.

In [ ]:
# ── Install all required packages ───────────────────────────────────────────
import subprocess, sys

PACKAGES = [
    'mftool',        # AMFI NAV data
    # mfapi.in is called directly through requests below; it is not a package dependency.
    'yfinance>=1.4.0', # Current release; includes documented network retry configuration
    'yahooquery',    # Yahoo Finance extended
    'mstarpy',       # Morningstar
    'jugaad-data',   # NSE index data (jugaad_data)
    'nselib',        # NSE library
    'nsepython',     # Another NSE helper
    'nsetools',      # NSE tools
    'requests',      # HTTP
    'pandas',        # DataFrames
    'numpy',         # Numerics
    'matplotlib',    # Plotting
    'plotly',        # Interactive plots
    'tabulate',      # Pretty table printing
    'python-dateutil', # Date parsing
    'scipy',         # Stats
]

for pkg in PACKAGES:
    try:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '--quiet', pkg],
            stderr=subprocess.DEVNULL
        )
        print(f'  ✅ {pkg}')
    except Exception as e:
        print(f'  ❌ {pkg}: {e}')

In [ ]:
# ── Core imports ─────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import json
from datetime import datetime, timedelta, date
from tabulate import tabulate

# Plotting defaults
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# ── Shared test fund identifiers ──────────────────────────────────────────────
# We test all sources against the same well-known fund to make comparison easy.

FUND = {
    'name':        'Quant Small Cap Fund - Direct Plan - Growth',
    'amfi_code':   '120828',          # AMFI / mftool / mfapi.in scheme code
    'isin':        'INF966L01131',    # ISIN for Direct Growth option
    'yahoo_tick':  '0P0001RQ6R.BO',  # Yahoo Finance ticker
    'mstar_id':    'F00000PDX2',      # Morningstar Fund ID
    'mstar_country': 'IN',
}

# A second fund for comparison tests
FUND2 = {
    'name':       'Parag Parikh Flexi Cap Fund - Direct Plan - Growth',
    'amfi_code':  '122639',
    'isin':       'INF879O01019',
    'yahoo_tick': '0P0001BKSG.BO',
    'mstar_id':   'F00001BGOU',
    'mstar_country': 'IN',
}

print('✅ Setup complete')
print(f'\nTest Fund 1: {FUND["name"]}')
print(f'Test Fund 2: {FUND2["name"]}')

---
## 2. mftool

**Source:** https://github.com/NayakwadiS/mftool  
**What it is:** Python wrapper around AMFI's publicly available data feed (`https://www.amfiindia.com/spages/NAVAll.txt`).  
**Expected data:** Scheme list, current NAV, historical NAV, fund house, category, scheme type.

### 2.1 Install check & basic fund info

In [ ]:
import mftool
mf = mftool.Mftool()
print('mftool version:', mftool.__version__ if hasattr(mftool, '__version__') else 'unknown')

In [ ]:
# ── 2.1 Fund basic info ───────────────────────────────────────────────────────
print('=== mftool: Basic Fund Info ===')

quote   = mf.get_scheme_quote(FUND['amfi_code'])
details = mf.get_scheme_details(FUND['amfi_code'])

print(f"Scheme Name    : {quote.get('scheme_name', 'N/A')}")
print(f"Fund House     : {details.get('fund_house', 'N/A')}")
print(f"Scheme Type    : {details.get('scheme_type', 'N/A')}")
print(f"Scheme Category: {details.get('scheme_category', 'N/A')}")
print(f"Inception Date : {details.get('scheme_start_date', 'N/A')}")
print(f"Current NAV    : ₹{quote.get('nav', 'N/A')} (as of {quote.get('date', 'N/A')})")
print()
print('Raw quote dict:')
print(json.dumps(quote, indent=2))
print('\nRaw details dict:')
print(json.dumps(details, indent=2))

In [ ]:
# ── 2.2 Historical NAV ────────────────────────────────────────────────────────
print('=== mftool: Historical NAV ===')

hist = mf.get_scheme_historical_nav(FUND['amfi_code'])
df_mftool = pd.DataFrame(hist['data'])
df_mftool['date'] = pd.to_datetime(df_mftool['date'], format='%d-%m-%Y')
df_mftool['nav']  = pd.to_numeric(df_mftool['nav'], errors='coerce')
df_mftool = df_mftool.sort_values('date').reset_index(drop=True)

print(f"Total rows     : {len(df_mftool)}")
print(f"Date range     : {df_mftool['date'].min().date()} → {df_mftool['date'].max().date()}")
print(f"NAV range      : ₹{df_mftool['nav'].min():.2f} → ₹{df_mftool['nav'].max():.2f}")
print()
print('Last 5 rows:')
print(df_mftool.tail())

# Plot NAV history
fig, ax = plt.subplots()
ax.plot(df_mftool['date'], df_mftool['nav'], linewidth=1.5, color='steelblue')
ax.set_title(f'mftool — {FUND["name"]} — Full NAV History')
ax.set_xlabel('Date')
ax.set_ylabel('NAV (₹)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.3 All available schemes (scheme universe) ───────────────────────────────
print('=== mftool: Scheme Universe ===')

# Get schemes for the fund house
amc_name = details.get('fund_house', '')
all_schemes = mf.get_available_schemes(amc_name)
print(f'Total schemes available for {amc_name}: {len(all_schemes)}')

# Preview as DataFrame
df_schemes = pd.DataFrame(list(all_schemes.items()), columns=['amfi_code', 'scheme_name'])
print('\nFirst 10 schemes:')
print(df_schemes.head(10).to_string(index=False))

In [ ]:
# ── 2.4 Search schemes ────────────────────────────────────────────────────────
print('=== mftool: Search Schemes ===')

# Search by fund name keyword
results = mf.get_scheme_codes("Parag Parikh")
print('Search "Parag Parikh":')
for code, name in list(results.items())[:10]:
    print(f'  {code}: {name}')

In [ ]:
# ── 2.5 Average NAV (AMFI average across all schemes) ────────────────────────
print('=== mftool: Average NAV Across All Schemes ===')

try:
    avg = mf.get_average_nav(FUND['amfi_code'])
    print(json.dumps(avg, indent=2))
except Exception as e:
    print(f'get_average_nav not available: {e}')

# Document what mftool CAN and CANNOT provide
print()
print('=== mftool Capability Summary ===')
capabilities = [
    ('Scheme Universe (all AMFI codes)','✅'),
    ('Current NAV',                     '✅'),
    ('Historical NAV (full since inception)', '✅'),
    ('Fund house / scheme type / category',  '✅'),
    ('Inception date',                  '✅ (embedded in first NAV row)'),
    ('Portfolio holdings',              '❌ Not available'),
    ('Expense ratio',                   '❌ Not available'),
    ('Exit load',                       '❌ Not available'),
    ('Risk metrics (Sharpe, Beta, etc.)','❌ Not available'),
    ('Fund manager info',               '❌ Not available'),
    ('AUM',                             '❌ Not available'),
    ('Benchmark mapping',               '❌ Not available'),
]
print(tabulate(capabilities, headers=['Data Point', 'Available?'], tablefmt='github'))

### mftool — Key Findings

| Aspect | Notes |
|--------|-------|
| **Best for** | Full historical NAV since inception, scheme universe listing |
| **Scheme count** | ~20,000+ (all AMFI-registered schemes) |
| **Data freshness** | Updated daily by AMFI |
| **Gap** | No holdings, no expense ratio, no fund manager, no risk metrics |
| **Rating** | ⭐⭐⭐⭐ (for NAV data) |

---
## 3. mfapi.in

**Source:** https://api.mfapi.in  
**What it is:** Free REST JSON API for Indian mutual fund data.  
**Endpoints:** `/mf/{scheme_code}` for full history, `/mf/search?q=` for name search.

In [ ]:
# ── 3.1 Basic fund details ────────────────────────────────────────────────────
BASE_MFAPI = 'https://api.mfapi.in/mf/'

def mfapi_get(fund_id):
    r = requests.get(f'{BASE_MFAPI}{fund_id}', timeout=15)
    r.raise_for_status()
    return r.json()

data = mfapi_get(FUND['amfi_code'])

print('=== mfapi.in: Meta Info ===')
meta = data.get('meta', {})
for k, v in meta.items():
    print(f'  {k:25s}: {v}')

print(f'\nTotal NAV records: {len(data.get("data", []))}')
print('\nSample (latest 5):')
df_mfapi = pd.DataFrame(data['data'])
df_mfapi['date'] = pd.to_datetime(df_mfapi['date'], format='%d-%m-%Y')
df_mfapi['nav']  = pd.to_numeric(df_mfapi['nav'])
df_mfapi = df_mfapi.sort_values('date')
print(df_mfapi.tail(5).to_string(index=False))

In [ ]:
# ── 3.2 Search by name ────────────────────────────────────────────────────────
print('=== mfapi.in: Search ===')

search_resp = requests.get('https://api.mfapi.in/mf/search', params={'q': 'HDFC Flexi'}, timeout=10)
search_results = search_resp.json()
print(f'Results for "HDFC Flexi": {len(search_results)} funds')
for r in search_results[:8]:
    print(f"  {r.get('schemeCode','N/A'):>7}  {r.get('schemeName','N/A')}")

In [ ]:
# ── 3.3 Fetch NAV for a date range ────────────────────────────────────────────
print('=== mfapi.in: NAV for a date range (last 1 year) ===')

end = date.today()
start = end - timedelta(days=365)

df_1yr = df_mfapi[df_mfapi['date'].dt.date >= start].copy()
print(f'Records in last 1 year: {len(df_1yr)}')
print(f'1-year return: {((df_1yr["nav"].iloc[-1] / df_1yr["nav"].iloc[0]) - 1) * 100:.2f}%')

# Plot
fig, ax = plt.subplots()
ax.plot(df_1yr['date'], df_1yr['nav'], color='darkorange')
ax.set_title('mfapi.in — Last 1-Year NAV')
ax.set_xlabel('Date')
ax.set_ylabel('NAV (₹)')
plt.tight_layout()
plt.show()

print()
print('=== mfapi.in Capability Summary ===')
cap2 = [
    ('Scheme code / name / fund house / type / category', '✅'),
    ('Full NAV history (since inception)',                '✅'),
    ('Search by fund name',                              '✅'),
    ('Holdings / portfolio',                             '❌'),
    ('Expense ratio / loads',                            '❌'),
    ('Risk metrics',                                     '❌'),
    ('Fund manager',                                     '❌'),
    ('AUM',                                              '❌'),
]
print(tabulate(cap2, headers=['Data Point', 'Available?'], tablefmt='github'))

### mfapi.in — Key Findings

| Aspect | Notes |
|--------|-------|
| **Best for** | NAV history, scheme search by name |
| **Advantage over mftool** | REST API (easier to cache, no Python dependency) |
| **Gap** | Same as mftool — no enrichment data |
| **Rating** | ⭐⭐⭐⭐ |

---
## 4. mf.captnemo.in

**Source:** https://github.com/captn3m0/mf.captnemo.in  
**What it is:** Kuvera API proxy — provides richer metadata including expense ratio, loads, investment options.  
**Endpoints:** `/kuvera/{isin}` for fund info, `/nav/{isin}` for NAV.

In [ ]:
# ── 4.1 Fund details via ISIN ────────────────────────────────────────────────
BASE_CAPTNEMO = 'https://mf.captnemo.in'

def captnemo_fund(isin):
    """Fetch fund details from mf.captnemo.in by ISIN.
    
    NOTE: The API may return a list (multiple plans) or a dict (single plan).
    We handle both cases robustly.
    """
    r = requests.get(f'{BASE_CAPTNEMO}/kuvera/{isin}', timeout=15)
    r.raise_for_status()
    raw = r.json()
    
    if isinstance(raw, list):
        print(f'  Note: API returned a list with {len(raw)} plan(s)')
        print(f'  Plans: {[p.get("plan","?") + "/" + ("Direct" if p.get("direct") else "Regular") for p in raw[:5]]}')
        # Prefer Direct Growth plan if available
        direct_growth = [p for p in raw 
                         if p.get('direct') and str(p.get('plan','')).lower() == 'growth']
        if direct_growth:
            print(f'  Selecting Direct Growth plan')
            return direct_growth[0]
        return raw[0]  # fallback to first
    
    return raw

# Using Edelweiss Banking PSU Debt (from docs example)
isin_test = 'INF843K01FC8'

print('=== mf.captnemo.in: Fund Info (Edelweiss Banking PSU Debt) ===')
fund_info = captnemo_fund(isin_test)
print(f'\nTop-level keys: {list(fund_info.keys())}')
print()
print(json.dumps(fund_info, indent=2, default=str)[:3000])  # Truncate


In [ ]:
# ── 4.2 Parse key fields ──────────────────────────────────────────────────────
print('=== mf.captnemo.in: Key Fields ===')

fields = [
    ('Code (Kuvera)',     fund_info.get('code')),
    ('Name',             fund_info.get('name')),
    ('Short Name',       fund_info.get('short_name')),
    ('Fund House',       fund_info.get('amc_code')),
    ('Category',         fund_info.get('category')),
    ('Fund Type',        fund_info.get('type')),
    ('Plan (Direct)',     fund_info.get('direct')),
    ('Growth/IDCW',      fund_info.get('plan')),
    ('Expense Ratio',    fund_info.get('expense_ratio')),
    ('AUM (₹ Cr)',       fund_info.get('aum')),
    ('NAV',              fund_info.get('nav')),
    ('Lock-in (days)',   fund_info.get('lock_in')),
    ('Maturity Type',    fund_info.get('maturity_type')),
    ('SIP Allowed',      fund_info.get('sip_flag')),
    ('STP Allowed',      fund_info.get('stp_flag')),
    ('SWP Allowed',      fund_info.get('swp_flag')),
    ('Redemption',       fund_info.get('redemption_allowed')),
    ('CRISIL Rating',    fund_info.get('crisil_rating')),
    ('Returns 1W',       fund_info.get('returns', {}).get('1week') if isinstance(fund_info.get('returns'), dict) else 'N/A'),
    ('Returns 1Y',       fund_info.get('returns', {}).get('1year') if isinstance(fund_info.get('returns'), dict) else 'N/A'),
    ('Returns 3Y',       fund_info.get('returns', {}).get('3year') if isinstance(fund_info.get('returns'), dict) else 'N/A'),
    ('Returns 5Y',       fund_info.get('returns', {}).get('5year') if isinstance(fund_info.get('returns'), dict) else 'N/A'),
]

print(tabulate(fields, headers=['Field', 'Value'], tablefmt='github'))

In [ ]:
# ── 4.3 Investment options details ────────────────────────────────────────────
print('=== mf.captnemo.in: Investment Options ===')

# Check for SIP details
purchase_info = fund_info.get('purchase', {})
sip_info      = fund_info.get('sip', {})

if purchase_info:
    print('Lump Sum Purchase:')
    for k, v in purchase_info.items():
        print(f'  {k}: {v}')

if sip_info:
    print('\nSIP Info:')
    print(json.dumps(sip_info, indent=2, default=str))

In [ ]:
# ── 4.4 NAV via ISIN ──────────────────────────────────────────────────────────
print('=== mf.captnemo.in: NAV ===')

try:
    nav_resp = requests.get(f'{BASE_CAPTNEMO}/nav/{isin_test}', timeout=15)
    nav_data = nav_resp.json()
    print(f'Type: {type(nav_data)}')
    if isinstance(nav_data, list):
        df_cap_nav = pd.DataFrame(nav_data)
        print(f'NAV records: {len(df_cap_nav)}')
        print(df_cap_nav.tail(5))
    else:
        print(json.dumps(nav_data, indent=2, default=str)[:500])
except Exception as e:
    print(f'Error: {e}')

print()
print('=== mf.captnemo.in Capability Summary ===')
cap4 = [
    ('Current NAV',                     '✅'),
    ('Fund house / type / category',    '✅'),
    ('Expense ratio',                   '✅'),
    ('AUM',                             '✅'),
    ('Lock-in period',                  '✅'),
    ('SIP / STP / SWP flags',           '✅'),
    ('Min investment (lumpsum & SIP)',   '✅'),
    ('CRISIL rating',                   '✅'),
    ('Returns (1W, 1Y, 3Y, 5Y)',        '✅ (pre-calculated)'),
    ('Exit load schedule',              '⚠️ Partial (may be in raw data)'),
    ('Full NAV history',                '⚠️ Limited'),
    ('Portfolio holdings',              '❌'),
    ('Fund manager details',            '❌'),
    ('Risk metrics (Sharpe, Beta)',     '⚠️ Volatility / Info Ratio only'),
    ('Benchmark mapping',               '❌'),
]
print(tabulate(cap4, headers=['Data Point', 'Available?'], tablefmt='github'))

### mf.captnemo.in — Key Findings

| Aspect | Notes |
|--------|-------|
| **Best for** | Expense ratio, AUM, min investment, SIP/STP/SWP flags, pre-calculated trailing returns |
| **Gap** | Holdings, fund manager, full risk metrics |
| **Reliability** | Depends on Kuvera's API — may change without notice |
| **Rating** | ⭐⭐⭐⭐ (for enrichment metadata) |

---
## 5. yfinance

**Source:** https://github.com/ranaroussi/yfinance  
**What it is:** Unofficial Yahoo Finance wrapper.  
**Relevance:** Good for index data, some fund data; requires finding Yahoo ticker for Indian MFs.

In [ ]:
import yfinance as yf
from packaging.version import Version

print('yfinance version:', yf.__version__)

# yfinance 0.2.57 is affected by the false YFRateLimitError regression reported
# in upstream issue #2422. The changelog identifies fixes in 0.2.58/0.2.59.
MIN_FIXED_YFINANCE_VERSION = Version('0.2.59')
can_query_yahoo = Version(yf.__version__) >= MIN_FIXED_YFINANCE_VERSION
yahoo_rate_limited = False

def is_yahoo_rate_limit_error(error):
    message = str(error).lower()
    return type(error).__name__ == 'YFRateLimitError' or any(
        marker in message for marker in ('too many requests', 'rate limited', '429')
    )

if not can_query_yahoo:
    print('Update required: re-run the install cell, restart the kernel, and run this cell again.')
    print('yfinance 0.2.57 has a known false-rate-limit regression; install current yfinance>=1.4.0.')
elif hasattr(yf, 'config'):
    # Documented yfinance exponential backoff for transient network failures.
    yf.config.network.retries = 2
else:
    print('Upgrade to yfinance>=1.4.0 to enable documented network retries.')

# ── 5.1 Indian MF via Yahoo ticker ───────────────────────────────────────────
print('\n=== yfinance: Indian MF — Quant Small Cap ===')

ticker = FUND['yahoo_tick']
fund_yf = yf.Ticker(ticker)
info = None

if can_query_yahoo:
    try:
        # Avoid an application-level retry loop: .info is a relatively costly Yahoo endpoint.
        info = fund_yf.get_info()
        print('Successfully retrieved ticker info')
    except Exception as e:
        if is_yahoo_rate_limit_error(e):
            yahoo_rate_limited = True
            print('Yahoo rate-limited the metadata request. No additional .info loop will be sent.')
            print('Wait before re-running Yahoo cells; use AMFI/mfapi for Indian-fund NAV meanwhile.')
        else:
            print(f'Fund metadata unavailable: {e}')

if info is not None:
    important_keys = [
        'longName', 'shortName', 'category', 'fundFamily', 'totalAssets',
        'annualReportExpenseRatio', 'morningStarOverallRating', 'morningStarRiskRating',
        'fundInceptionDate', 'yield', 'lastDividendValue', 'lastDividendDate',
        'beta3Year', 'ytdReturn', 'threeYearAverageReturn', 'fiveYearAverageReturn'
    ]
    print('\nKey info fields:')
    rows = []
    for k in important_keys:
        v = info.get(k, 'N/A')
        if k == 'fundInceptionDate' and isinstance(v, int):
            v = datetime.fromtimestamp(v).strftime('%Y-%m-%d')
        rows.append((k, v))
    print(tabulate(rows, headers=['Field', 'Value'], tablefmt='github'))
else:
    print('\nMetadata unavailable. Indian MF coverage on Yahoo may be incomplete.')

In [ ]:
# ── 5.2 Historical price (NAV) data ──────────────────────────────────────────
print('=== yfinance: Historical NAV ===')

if not can_query_yahoo:
    print('Skipping Yahoo call until yfinance has been updated and the kernel restarted.')
elif yahoo_rate_limited:
    print('Skipping Yahoo call after the rate-limited metadata request; retry later.')
else:
    try:
        hist_yf = fund_yf.history(period='max')
        print(f'Total rows : {len(hist_yf)}')
        print(f'Columns    : {list(hist_yf.columns)}')
        if not hist_yf.empty:
            print(f'Date range : {hist_yf.index.min().date()} → {hist_yf.index.max().date()}')
            print('\nLast 5 rows:')
            print(hist_yf[['Close']].tail())
        else:
            print('No historical data available for this ticker.')
    except Exception as e:
        if is_yahoo_rate_limit_error(e):
            yahoo_rate_limited = True
            print('Yahoo rate-limited the history request. Retry later or use AMFI/mfapi NAV data.')
        else:
            print(f'Error: {e}')

In [ ]:
# ── 5.3 NIFTY 500 Index — primary benchmark ───────────────────────────────────
print('=== yfinance: NIFTY 500 Index (benchmark) ===')

# Common Indian indices on Yahoo Finance
indices = {
    'NIFTY 500':      '^CRSLDX',       # Note: may need verification
    'NIFTY 50':       '^NSEI',
    'NIFTY Midcap150':'NIFTYMIDCAP150.NS',
    'S&P 500':        '^GSPC',         # For comparison
    'NASDAQ 100':     '^NDX',          # For international/FOF funds
}

if not can_query_yahoo or yahoo_rate_limited:
    print('Skipping benchmark calls because Yahoo querying is unavailable for this run.')
    indices = {}

for name, sym in indices.items():
    try:
        ticker_obj = yf.Ticker(sym)
        hist = ticker_obj.history(period='1y')
        if not hist.empty:
            one_yr_ret = ((hist['Close'].iloc[-1] / hist['Close'].iloc[0]) - 1) * 100
            print(f'{name:25s} ({sym:25s}): 1Y return = {one_yr_ret:+.2f}%, rows={len(hist)}')
        else:
            print(f'{name:25s} ({sym:25s}): ❌ No data')
    except Exception as e:
        if is_yahoo_rate_limit_error(e):
            yahoo_rate_limited = True
            print('Yahoo rate-limited benchmark history; stopping further Yahoo calls.')
            break
        print(f'{name:25s}: ❌ {e}')

In [ ]:
# ── 5.4 NIFTY 50 full history ────────────────────────────────────────────────
print('=== yfinance: NIFTY 50 Full History ===')

nifty50 = pd.DataFrame()
if not can_query_yahoo or yahoo_rate_limited:
    print('Skipping NIFTY 50 call because Yahoo querying is unavailable for this run.')
else:
    try:
        nifty50 = yf.Ticker('^NSEI').history(period='max')
    except Exception as e:
        if is_yahoo_rate_limit_error(e):
            yahoo_rate_limited = True
            print('Yahoo rate-limited the NIFTY 50 request; stopping further Yahoo calls.')
        else:
            print(f'Error: {e}')

if not nifty50.empty:
    print(f'Rows: {len(nifty50)}')
    print(f'Date range: {nifty50.index.min().date()} → {nifty50.index.max().date()}')
    print(nifty50[['Close']].tail(5))

    fig, ax = plt.subplots()
    ax.plot(nifty50.index, nifty50['Close'], color='green', linewidth=1)
    ax.set_title('NIFTY 50 — Full History via yfinance')
    ax.set_xlabel('Date')
    ax.set_ylabel('Index Value')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 5.5 Ticker search / lookup ────────────────────────────────────────────────
print('=== yfinance: Ticker Search (new feature in recent versions) ===')

if not can_query_yahoo or yahoo_rate_limited:
    print('Skipping ticker search because Yahoo querying is unavailable for this run.')
else:
    try:
        search = yf.Search('Quant Small Cap', max_results=5)
        quotes = search.quotes
        print(f'Found {len(quotes)} results')
        for q in quotes:
            print(f"  {q.get('symbol','N/A'):20s} {q.get('longname','N/A')}")
    except Exception as e:
        if is_yahoo_rate_limit_error(e):
            yahoo_rate_limited = True
            print('Yahoo rate-limited the search request; stopping further Yahoo calls.')
        else:
            print(f'Search not available or error: {e}')

print()
print('=== yfinance Capability Summary ===')
cap5 = [
    ('Indian MF NAV history (via Yahoo ticker)',    '✅ (if ticker found)'),
    ('Indian benchmark indices (NIFTY 50, etc.)',   '✅'),
    ('International indices (S&P500, NASDAQ)',      '✅'),
    ('Fund basic info (name, category, AUM)',       '⚠️ Often N/A for Indian MFs'),
    ('Expense ratio',                               '⚠️ Often 0% or N/A'),
    ('Morningstar ratings',                         '⚠️ Sometimes available'),
    ('Holdings (via .info)',                        '❌ Not for Indian MFs'),
    ('Risk metrics (Beta, Sharpe)',                 '⚠️ Limited'),
    ('Dividends',                                   '✅ (if declared)'),
    ('All Indian MF schemes',                       '❌ Many missing tickers'),
]
print(tabulate(cap5, headers=['Data Point', 'Available?'], tablefmt='github'))

---
## 6. yahooquery

**Source:** https://github.com/dpguthrie/yahooquery  
**What it is:** More complete Yahoo Finance wrapper — covers portfolio holdings data that `yfinance` misses.

In [ ]:
from yahooquery import Ticker as YQTicker
import yahooquery as yq

print('yahooquery version:', yq.__version__)

# Initialize
fund_yq = YQTicker(FUND['yahoo_tick'])
ticker  = FUND['yahoo_tick']

In [ ]:
# ── 6.1 Summary detail ────────────────────────────────────────────────────────
print('=== yahooquery: Summary Detail ===')

try:
    sd = fund_yq.summary_detail
    if ticker in sd:
        detail = sd[ticker]
        print(json.dumps(detail, indent=2, default=str))
    else:
        print(f'No summary detail for {ticker}:', sd)
except Exception as e:
    print(f'Error: {e}')

In [ ]:
# ── 6.2 Fund performance data ─────────────────────────────────────────────────
print('=== yahooquery: Fund Performance ===')

try:
    fp = fund_yq.fund_performance
    if ticker in fp:
        perf = fp[ticker]
        print(json.dumps(perf, indent=2, default=str)[:3000])
    else:
        print('Fund performance:', fp)
except Exception as e:
    print(f'Error: {e}')

In [ ]:
# ── 6.3 Portfolio holdings ────────────────────────────────────────────────────
print('=== yahooquery: Portfolio Holdings (Top Holdings) ===')

try:
    holdings = fund_yq.fund_top_holdings
    print(f'Type: {type(holdings)}')
    if isinstance(holdings, pd.DataFrame) and not holdings.empty:
        print(f'Columns: {list(holdings.columns)}')
        print(f'Records: {len(holdings)}')
        print(holdings.to_string(index=False))
    else:
        print('No holdings data:', holdings)
except Exception as e:
    print(f'Error: {e}')

In [ ]:
# ── 6.4 Sector weightings ─────────────────────────────────────────────────────
print('=== yahooquery: Sector Weightings ===')

try:
    sectors = fund_yq.fund_sector_weightings
    print(f'Type: {type(sectors)}')
    if isinstance(sectors, pd.DataFrame) and not sectors.empty:
        print(sectors.to_string())
    else:
        print('No sector data:', sectors)
except Exception as e:
    print(f'Error: {e}')

In [ ]:
# ── 6.5 Asset profile ─────────────────────────────────────────────────────────
print('=== yahooquery: Asset Profile ===')

try:
    profile = fund_yq.asset_profile
    if ticker in profile:
        print(json.dumps(profile[ticker], indent=2, default=str))
    else:
        print('Asset profile:', profile)
except Exception as e:
    print(f'Error: {e}')

In [ ]:
# ── 6.6 Fund holding info (bond rating, market cap blend) ─────────────────────
print('=== yahooquery: Fund Holding Info ===')

try:
    fhi = fund_yq.fund_holding_info
    if ticker in fhi:
        print(json.dumps(fhi[ticker], indent=2, default=str))
    else:
        print('Fund holding info:', fhi)
except Exception as e:
    print(f'Error: {e}')

print()
print('=== yahooquery Capability Summary ===')
cap6 = [
    ('Historical NAV (price history)',               '✅'),
    ('Portfolio top holdings (symbol + weight)',     '✅ (for mapped funds)'),
    ('Sector weightings',                            '✅'),
    ('Market cap blend (large/mid/small)',           '✅'),
    ('Annual / quarterly returns',                   '✅ (fund_performance)'),
    ('52-week high/low',                             '✅'),
    ('Fund performance vs benchmark',                '⚠️ Partial'),
    ('Expense ratio',                                '⚠️ Often 0 for Indian'),
    ('AUM',                                          '⚠️ Partial'),
    ('Fund manager',                                 '❌'),
    ('All Indian MF schemes',                        '❌ Ticker mapping required'),
]
print(tabulate(cap6, headers=['Data Point', 'Available?'], tablefmt='github'))

---
## 7. mstarpy — Morningstar

**Source:** https://github.com/Mael-J/mstarpy  
**What it is:** Python wrapper for Morningstar's public (non-authenticated) data.  
**Expected data:** Returns, risk metrics, holdings, sector, Morningstar ratings.

In [ ]:
import mstarpy
import inspect
print('mstarpy version:', mstarpy.__version__ if hasattr(mstarpy, '__version__') else 'unknown')

# ── Detect correct Funds() constructor signature (varies by mstarpy version) ────
# Older versions: Funds(term=..., country='IN')
# Newer versions: Funds(term=...) only, country dropped

def init_mstarpy_fund(term, country='IN'):
    """Version-agnostic mstarpy Funds initializer."""
    sig = inspect.signature(mstarpy.Funds.__init__)
    params = list(sig.parameters.keys())
    print(f'mstarpy.Funds params: {params}')
    
    if 'country' in params:
        return mstarpy.Funds(term=term, country=country)
    elif 'region' in params:
        # Map country codes to region
        region_map = {'IN': 'ASIA', 'US': 'USA', 'GB': 'GBR'}
        return mstarpy.Funds(term=term, region=region_map.get(country, 'ASIA'))
    else:
        # Try without country
        try:
            return mstarpy.Funds(term=term)
        except Exception:
            # Last resort: search and init
            results = mstarpy.search_funds('quant small cap', pageSize=5)
            if results:
                found = next((r for r in results if 'small cap' in r.get('Name','').lower()), results[0])
                print(f'Found via search: {found.get("Name","N/A")}')
                return mstarpy.Funds(term=found.get('SecId', term))
            raise ValueError('Could not initialize mstarpy.Funds')

TERM    = FUND['mstar_id']
COUNTRY = FUND['mstar_country']

try:
    fund_ms = init_mstarpy_fund(TERM, COUNTRY)
    print(f'\n✅ Fund initialized: {getattr(fund_ms, "name", fund_ms)}')
except Exception as e:
    print(f'\n❌ Error: {e}')
    fund_ms = None


In [ ]:
# ── 7.1 Trailing returns + category + Morningstar rating ─────────────────────
print('=== mstarpy: Trailing Returns & Ratings ===')

try:
    tr = fund_ms.trailingReturn()
    print(f'Type: {type(tr)}')
    if isinstance(tr, dict):
        print(f'Keys: {list(tr.keys())}')
        
        periods  = tr.get('columnDefs', [])
        fund_ret = tr.get('totalReturnNAV', [])
        cat_ret  = tr.get('totalReturnCategoryNew', [])
        ranks    = tr.get('returnRank', [])
        
        rows = []
        for i, p in enumerate(periods):
            fr = f"{fund_ret[i]:.2f}%" if i < len(fund_ret) and isinstance(fund_ret[i], (int,float)) else 'N/A'
            cr = f"{cat_ret[i]:.2f}%"  if i < len(cat_ret)  and isinstance(cat_ret[i],  (int,float)) else 'N/A'
            rk = str(ranks[i])         if i < len(ranks)    and ranks[i] is not None else 'N/A'
            rows.append((p, fr, cr, rk))
        print(tabulate(rows, headers=['Period','Fund','Category Avg','Rank'], tablefmt='github'))
        
        print(f"\nMorningstar Ratings:")
        print(f"  Overall : {'★' * (tr.get('overallMorningstarRating') or 0)} ({tr.get('overallMorningstarRating')} stars)")
        print(f"  3-Year  : {'★' * (tr.get('morningstarRatingFor3Year') or 0)}")
        print(f"  5-Year  : {'★' * (tr.get('morningstarRatingFor5Year') or 0)}")
        print(f"  10-Year : {'★' * (tr.get('morningstarRatingFor10Year') or 0)}")
        print(f"\nCategory: {tr.get('categoryName','N/A')}")
        print(f"Inception: {tr.get('inceptionDate','N/A')}")
except Exception as e:
    print(f'Error: {e}')

In [ ]:
# ── 7.2 Risk & Volatility ─────────────────────────────────────────────────────
print('=== mstarpy: Risk & Volatility ===')

def safe_extract(data, keys, default='N/A'):
    try:
        cur = data
        for k in keys:
            cur = cur[k]
        return cur if cur is not None else default
    except:
        return default

try:
    rv = fund_ms.riskVolatility()
    fund_rv = safe_extract(rv, ['fundRiskVolatility'])
    
    periods_risk = ['for3Year', 'for5Year', 'for10Year']
    metrics_risk = ['standardDeviation', 'sharpeRatio', 'beta', 'alpha', 'rSquared', 'informationRatio']
    
    rows = []
    for p in periods_risk:
        pdata = safe_extract(fund_rv, [p]) if isinstance(fund_rv, dict) else {}
        row = [p.replace('for','')]
        for m in metrics_risk:
            v = pdata.get(m, 'N/A') if isinstance(pdata, dict) else 'N/A'
            row.append(f'{v:.3f}' if isinstance(v, float) else str(v))
        rows.append(row)
    
    print(tabulate(rows, headers=['Period'] + metrics_risk, tablefmt='github'))
except Exception as e:
    print(f'Error: {e}')

In [ ]:
# ── 7.3 Max Drawdown ──────────────────────────────────────────────────────────
print('=== mstarpy: Max Drawdown ===')

try:
    md = fund_ms.maxDrawDown()
    print(json.dumps(md, indent=2, default=str)[:1500])
except Exception as e:
    print(f'Error: {e}')

In [ ]:
# ── 7.4 Holdings ──────────────────────────────────────────────────────────────
print('=== mstarpy: Portfolio Holdings ===')

try:
    holdings_eq = fund_ms.holdings(holdingType='equity')
    if isinstance(holdings_eq, pd.DataFrame) and not holdings_eq.empty:
        print(f'Total equity holdings: {len(holdings_eq)}')
        print(f'Columns: {list(holdings_eq.columns)}')
        print('\nTop 15 holdings:')
        print(holdings_eq.head(15)[['securityName','weighting','sector','country']].to_string(index=False))
        
        # Concentration metrics
        print(f"\nTop 5  weight: {holdings_eq.head(5)['weighting'].sum():.2f}%")
        print(f"Top 10 weight: {holdings_eq.head(10)['weighting'].sum():.2f}%")
        print(f"Total holdings: {len(holdings_eq)}")
        
        # Visualize top 10
        top10 = holdings_eq.head(10).set_index('securityName')['weighting']
        fig, ax = plt.subplots(figsize=(10, 6))
        top10.sort_values().plot(kind='barh', ax=ax, color='steelblue')
        ax.set_title('mstarpy — Top 10 Holdings (% weight)')
        ax.set_xlabel('Weight (%)')
        plt.tight_layout()
        plt.show()
    else:
        print(f'Holdings: {holdings_eq}')
except Exception as e:
    print(f'Error: {e}')

In [ ]:
# ── 7.5 Sector allocation ─────────────────────────────────────────────────────
print('=== mstarpy: Sector Allocation ===')

try:
    sector = fund_ms.sector()
    if isinstance(sector, dict) and 'EQUITY' in sector:
        portfolio = sector['EQUITY'].get('fundPortfolio', {})
        portfolio_date = portfolio.pop('portfolioDate', 'unknown')
        print(f'As of: {portfolio_date}')
        
        sec_rows = sorted(
            [(k.replace('_',' ').title(), v) for k,v in portfolio.items() if isinstance(v,(int,float))],
            key=lambda x: x[1], reverse=True
        )
        print(tabulate(sec_rows, headers=['Sector','Weight (%)'], tablefmt='github'))
        
        # Pie chart
        labels, sizes = zip(*[(s,w) for s,w in sec_rows if w > 0])
        fig, ax = plt.subplots(figsize=(8,6))
        ax.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
        ax.set_title('mstarpy — Sector Allocation')
        plt.tight_layout()
        plt.show()
    else:
        print(f'Sector data: {sector}')
except Exception as e:
    print(f'Error: {e}')

In [ ]:
# ── 7.6 NAV history ───────────────────────────────────────────────────────────
print('=== mstarpy: Historical NAV ===')

try:
    end_dt   = date.today()
    start_dt = end_dt - timedelta(days=365*3)  # 3 years
    nav_ms = fund_ms.nav(start_date=start_dt, end_date=end_dt, frequency='daily')
    
    if nav_ms:
        df_nav_ms = pd.DataFrame(nav_ms)
        print(f'Records: {len(df_nav_ms)}')
        print(f'Columns: {list(df_nav_ms.columns)}')
        print('Last 5:')
        print(df_nav_ms.tail(5))
    else:
        print('No NAV data')
except Exception as e:
    print(f'Error: {e}')

In [ ]:
# ── 7.7 Fee level ─────────────────────────────────────────────────────────────
print('=== mstarpy: Fee Level ===')

try:
    fees = fund_ms.feeLevel()
    print(json.dumps(fees, indent=2, default=str))
except Exception as e:
    print(f'Error: {e}')

# ── 7.8 Search funds ──────────────────────────────────────────────────────────
print('\n=== mstarpy: Fund Search ===')

try:
    results = mstarpy.search_funds('flexi cap', country='IN', pageSize=10)
    print(f'Found {len(results)} funds')
    for r in results[:10]:
        print(f"  {r.get('SecId',''):15s} {r.get('Name','N/A')}")
except Exception as e:
    print(f'Error: {e}')

print()
print('=== mstarpy Capability Summary ===')
cap7 = [
    ('Trailing returns (vs category & rank)',    '✅'),
    ('Morningstar star ratings (3Y/5Y/10Y/overall)', '✅'),
    ('Risk metrics: Std dev, Sharpe, Beta, Alpha, R²', '✅'),
    ('Max drawdown',                             '✅'),
    ('Portfolio holdings (equity)',              '✅ (name, weight, sector, country)'),
    ('Bond holdings',                            '✅'),
    ('Sector allocation',                        '✅'),
    ('Historical NAV',                           '✅ (daily, up to 10Y)'),
    ('Category name',                            '✅'),
    ('Inception date',                           '✅'),
    ('Expense ratio',                            '⚠️ Not available for Indian funds'),
    ('AUM',                                      '⚠️ Not directly available'),
    ('Fund manager details',                     '⚠️ Limited'),
    ('Exit load / stamp duty',                   '❌'),
    ('Coverage for all Indian MFs',              '⚠️ Morningstar-covered funds only'),
]
print(tabulate(cap7, headers=['Data Point', 'Available?'], tablefmt='github'))

### mstarpy — Key Findings

| Aspect | Notes |
|--------|-------|
| **Best for** | Risk metrics, Morningstar ratings, portfolio holdings, sector data |
| **Unique** | Trailing returns WITH category average AND rank (very valuable!) |
| **Gap** | Expense ratio unavailable for Indian funds; not all funds covered |
| **Rating** | ⭐⭐⭐⭐⭐ (most data-rich source for analytics) |

---
## 8. NSE Index Data (jugaad_data, nselib, nsepython)

**Purpose:** Benchmark index history for Indian markets — crucial for performance comparison.

In [ ]:
# ── 8.1 jugaad_data ───────────────────────────────────────────────────────────
print('=== jugaad_data: NSE Index ===')
# jugaad_data v2+ changed its API significantly.
# The old `get_index_data()` is gone. We use `NSEHistory` with correct method.
try:
    from jugaad_data.nse import NSEHistory
    nse = NSEHistory()
    
    # Try the correct method signature for current version
    import inspect
    methods = [m for m in dir(nse) if not m.startswith('_')]
    print(f'NSEHistory available methods: {methods}')
    
    # Attempt equity/index data with the discovered method
    import datetime as dt_mod
    from_dt = dt_mod.date(2024, 1, 1)
    to_dt   = dt_mod.date(2024, 3, 31)
    
    if 'index_raw' in methods:
        raw = nse.index_raw('NIFTY 50', from_dt, to_dt)
        print(f'index_raw result type: {type(raw)}')
        if raw is not None:
            import pandas as pd
            df_jd = pd.DataFrame(raw)
            print(df_jd.tail(3))
    elif 'stock_raw' in methods:
        print('index_raw not available; showing stock_raw as demo:')
        raw = nse.stock_raw('TCS', from_dt, to_dt)
        if raw:
            import pandas as pd
            df_jd = pd.DataFrame(raw)
            print(f'stock_raw rows: {len(df_jd)}, cols: {list(df_jd.columns)[:6]}')
    else:
        print(f'Available methods: {methods}')
        print('NOTE: jugaad_data API has changed. Use yfinance for benchmark data instead.')
except ImportError:
    print('jugaad_data not installed — skipping')
except Exception as e:
    print(f'jugaad_data error: {e}')
    print('NOTE: jugaad_data API is unstable. Recommended: use yfinance for benchmarks.')

print()
print('VERDICT: jugaad_data API changes frequently. Not recommended for production.')
print('         Use yfinance (^NSEI, ^NSEBANK) or NSE direct API instead.')


In [ ]:
# ── 8.2 nselib ────────────────────────────────────────────────────────────────
print('=== nselib: NSE Data ===')
# nselib API changed — index_data() no longer accepts start_date/end_date kwargs.
# Correct usage (current version):
try:
    import nselib
    from nselib import capital_market
    print('nselib imported successfully')
    
    # Show available functions
    import inspect
    fns = [f for f in dir(capital_market) if not f.startswith('_')]
    print(f'capital_market functions: {fns[:15]}')
    
    # Try the correct function signature (no start_date/end_date)
    try:
        # Current API: equity_list() for stock list
        eq_list = capital_market.equity_list()
        if eq_list is not None:
            print(f'equity_list: {type(eq_list)}, len={len(eq_list) if hasattr(eq_list,"__len__") else "?"}')
    except Exception as e2:
        print(f'equity_list error: {e2}')

except ImportError:
    print('nselib not installed')
except Exception as e:
    print(f'nselib error: {e}')

print()
print('VERDICT: nselib useful for equity data but NSE API changes break it.')
print('         For benchmark indices: use yfinance or NSE direct API.')


In [ ]:
# ── 8.3 nsepython ─────────────────────────────────────────────────────────────
print('=== nsepython: NSE Index History ===')
try:
    import nsepython
    print('nsepython version:', nsepython.__version__ if hasattr(nsepython, '__version__') else '0.1')
    
    # Show available functions
    fns = [f for f in dir(nsepython) if not f.startswith('_') and callable(getattr(nsepython, f))]
    print(f'Available functions (sample): {fns[:12]}')
    
    # Try equity_history (works in most versions)
    try:
        eq = nsepython.equity_history('TCS', 'EQ', '01-01-2024', '31-03-2024')
        print(f'equity_history (TCS) rows: {len(eq) if eq is not None else "None"}')
    except Exception as e2:
        print(f'equity_history error: {e2}')
    
    # Try expiry_list
    try:
        exp = nsepython.expiry_list('NIFTY', 'options')
        print(f'expiry_list (NIFTY options): {exp[:3] if exp is not None else "None"}')
    except Exception as e3:
        print(f'expiry_list error: {e3}')

except ImportError:
    print('nsepython not installed')
except Exception as e:
    print(f'nsepython error: {e}')

print()
print('VERDICT: nsepython useful for F&O/options data, not ideal for historical index data.')
print('         Recommended: Use NSE direct API (Section 8.4) or yfinance for indices.')


In [ ]:
# ── 8.4 NSE Direct API (no library) ──────────────────────────────────────────
print('=== NSE Direct API: Index History ===')

# NSE provides data via its official APIs — test them directly
nse_headers = {
    'User-Agent': 'Mozilla/5.0',
    'Accept': 'application/json, text/javascript, */*; q=0.01',
    'Referer': 'https://www.nseindia.com/'
}

def nse_session_get(url):
    """NSE requires a cookie session — establish one first."""
    session = requests.Session()
    session.get('https://www.nseindia.com/', headers=nse_headers, timeout=10)
    session.get('https://www.nseindia.com/market-data/live-equity-market', headers=nse_headers, timeout=10)
    r = session.get(url, headers=nse_headers, timeout=15)
    return r

try:
    # Index info
    r = nse_session_get('https://www.nseindia.com/api/allIndices')
    if r.status_code == 200:
        data = r.json()
        indices = data.get('data', [])
        print(f'Available indices: {len(indices)}')
        print('Sample:')
        for idx in indices[:10]:
            print(f"  {idx.get('index','N/A'):35s} : {idx.get('last','N/A')} ({idx.get('variation','N/A')}%)")
    else:
        print(f'Status: {r.status_code}')
except Exception as e:
    print(f'NSE Direct API error: {e}')

In [ ]:
# ── 8.5 Benchmark via yfinance (most reliable fallback) ──────────────────────
print('=== Benchmark Indices via yfinance (recommended fallback) ===')

BENCHMARK_MAP = {
    'NIFTY 50':               '^NSEI',
    'NIFTY 500':              '^CRSLDX',
    'NIFTY Midcap 150':       'NIFMID150.NS',
    'NIFTY Smallcap 250':     'NIFSMCP250.NS',
    'NIFTY Next 50':          '^NSMIDCP',
    'NIFTY Bank':             '^NSEBANK',
    'NASDAQ 100':             '^NDX',
    'S&P 500':                '^GSPC',
    'CRISIL Liquid Index':    None,  # Not on Yahoo
    'CRISIL Short Term Bond': None,  # Not on Yahoo
}

results = []
if not can_query_yahoo or yahoo_rate_limited:
    print('Skipping Yahoo benchmark calls because querying is unavailable for this run.')
    BENCHMARK_MAP = {}

for name, sym in BENCHMARK_MAP.items():
    if sym is None:
        results.append((name, 'N/A', '❌ Not on Yahoo Finance', 'N/A'))
        continue
    try:
        h = yf.Ticker(sym).history(period='1y')
        if not h.empty:
            ret = ((h['Close'].iloc[-1]/h['Close'].iloc[0])-1)*100
            results.append((name, sym, '✅', f'{ret:+.2f}%'))
        else:
            results.append((name, sym, '⚠️ No data', 'N/A'))
    except Exception as e:
        results.append((name, sym, f'❌ {e}', 'N/A'))

print(tabulate(results, headers=['Benchmark','Yahoo Symbol','Status','1Y Return'], tablefmt='github'))

---
## 9. AMFI Direct API — Scheme Universe

**Source:** https://www.amfiindia.com  
**What it is:** The official AMFI feed — the authoritative source for all registered Indian MF schemes.

In [ ]:
# ── 9.1 AMFI NAVAll feed ──────────────────────────────────────────────────────
print('=== AMFI Direct: NAVAll Feed ===')

amfi_url = 'https://www.amfiindia.com/spages/NAVAll.txt'

try:
    r = requests.get(amfi_url, timeout=30)
    lines = r.text.strip().split('\n')
    print(f'Total lines: {len(lines)}')
    print('First 15 lines:')
    for l in lines[:15]:
        print(f'  {l}')
except Exception as e:
    print(f'Error: {e}')

In [ ]:
# ── 9.2 Parse AMFI feed into a clean DataFrame ────────────────────────────────
print('=== AMFI: Parsing Scheme Universe ===')

def parse_amfi_feed(text):
    records = []
    current_amc = ''
    current_category = ''
    
    for line in text.strip().split('\n'):
        line = line.strip()
        if not line:
            continue
        # AMC header lines (no semicolons)
        if ';' not in line:
            if 'Mutual Fund' in line or 'AMC' in line or 'Asset' in line:
                current_amc = line
            continue
        
        parts = line.split(';')
        if len(parts) >= 6 and parts[0].strip().isdigit():
            records.append({
                'amfi_code':     parts[0].strip(),
                'isin_growth':   parts[1].strip(),
                'isin_idcw':     parts[2].strip(),
                'scheme_name':   parts[3].strip(),
                'nav':           parts[4].strip(),
                'date':          parts[5].strip(),
                'amc':           current_amc,
            })
    
    return pd.DataFrame(records)

df_amfi = parse_amfi_feed(r.text)
df_amfi['nav'] = pd.to_numeric(df_amfi['nav'], errors='coerce')

print(f'Total schemes: {len(df_amfi)}')
print(f'Columns: {list(df_amfi.columns)}')
print('\nSample:')
print(df_amfi.head(10).to_string(index=False))

In [ ]:
# ── 9.3 Scheme statistics ─────────────────────────────────────────────────────
print('=== AMFI: Scheme Statistics ===')

print(f'Total unique schemes       : {len(df_amfi)}')
print(f'Schemes with valid NAV     : {df_amfi["nav"].notna().sum()}')
print(f'Schemes with Growth ISIN   : {(df_amfi["isin_growth"] != "-").sum()}')
print(f'Schemes with IDCW ISIN     : {(df_amfi["isin_idcw"] != "-").sum()}')

# Filter for Direct Growth plans only
direct_growth = df_amfi[df_amfi['scheme_name'].str.contains(
    'Direct', case=False, na=False
) & df_amfi['scheme_name'].str.contains(
    'Growth', case=False, na=False
)]
print(f'\nDirect Growth plans        : {len(direct_growth)}')
print('\nSample Direct Growth plans:')
print(direct_growth[['amfi_code','scheme_name','nav']].head(10).to_string(index=False))

In [ ]:
# ── 9.4 Historical NAV from AMFI ──────────────────────────────────────────────
print('=== AMFI: Historical NAV API ===')

# AMFI also has a historical NAV endpoint
def amfi_historical_nav(scheme_code, start_date='01-01-2020', end_date=None):
    if end_date is None:
        end_date = date.today().strftime('%d-%m-%Y')
    url = 'https://portal.amfiindia.com/DownloadNAVHistoryReport_Po.aspx'
    params = {
        'frmdt': start_date,
        'todt':  end_date,
        'mf':    scheme_code,  # fund code (sometimes AMC code, differs from AMFI scheme code)
    }
    r = requests.get(url, params=params, timeout=15)
    return r.text

# Note: Also try the MF scheme NAV download endpoint
def amfi_nav_by_scheme(amfi_code, from_date, to_date):
    url = 'https://www.amfiindia.com/modules/NAVHistoryReport_Po_MF.aspx'
    params = {
        'mf':    '',
        'tp':    'S',  # Scheme level
        'scheme': amfi_code,
        'frmdt':  from_date,
        'todt':   to_date,
    }
    r = requests.get(url, params=params, timeout=15, headers={'User-Agent': 'Mozilla/5.0'})
    return r.text

print('AMFI NAV feed format (the mftool / mfapi.in both read from this source):')
print('  URL: https://www.amfiindia.com/spages/NAVAll.txt')
print('  Format: AMC_Name; ISIN_Growth; ISIN_IDCW; Scheme_Name; NAV; Date')
print('  Updated: Daily (post-market close)')
print('  Coverage: All SEBI-registered open-ended schemes')

---
## 10. IndianAPI.in Exploration

**Source:** https://indianapi.in/  
**What it is:** Commercial API aggregator with Indian market data — stocks, mutual funds, and more.  
**Note:** Requires API key for most endpoints.

In [ ]:
# ── 10.1 Explore free / trial endpoints ──────────────────────────────────────
print('=== IndianAPI.in: Exploring Free Endpoints ===')

# Note: API key required for full access
# Set your API key here if you have one:
INDIAN_API_KEY = ''  # Replace with your key

if not INDIAN_API_KEY:
    print('⚠️  No API key set. Showing documented endpoints only.')
    print()
    
    endpoints = [
        ('GET /mutual_fund/search',          'Search mutual funds by name/keyword'),
        ('GET /mutual_fund/category',        'List all MF categories'),
        ('GET /mutual_fund/{code}',          'Fund details by AMFI code'),
        ('GET /mutual_fund/nav/{code}',      'NAV history'),
        ('GET /mutual_fund/holdings/{code}', 'Portfolio holdings'),
        ('GET /stock/search',                'Stock search'),
        ('GET /stock/quote/{symbol}',        'Stock quote'),
        ('GET /index/list',                  'Market indices'),
        ('GET /index/history/{name}',        'Index history'),
    ]
    print('Documented endpoints:')
    print(tabulate(endpoints, headers=['Endpoint','Description'], tablefmt='github'))
else:
    headers = {'X-Api-Key': INDIAN_API_KEY}
    
    # Test mutual fund search
    r = requests.get(
        'https://api.indianapi.in/mutual_fund/search',
        params={'q': 'Quant Small Cap'},
        headers=headers, timeout=10
    )
    print(f'Search response status: {r.status_code}')
    if r.status_code == 200:
        data = r.json()
        print(json.dumps(data, indent=2, default=str)[:2000])

---
## 11. BSE Mutual Fund Data

BSE (Bombay Stock Exchange) provides mutual fund NAV data and is a secondary source for validation.

In [ ]:
# ── 11.1 BSE MF NAV API ───────────────────────────────────────────────────────
print('=== BSE: Mutual Fund NAV ===')

bse_headers = {
    'User-Agent': 'Mozilla/5.0',
    'Accept': 'application/json',
}

try:
    # BSE MF scheme list
    r = requests.get(
        'https://api.bseindia.com/BseIndiaAPI/api/mfscheme/w',
        headers=bse_headers, timeout=10
    )
    print(f'BSE MF API status: {r.status_code}')
    if r.status_code == 200:
        data = r.json()
        print(json.dumps(data, indent=2, default=str)[:1000])
except Exception as e:
    print(f'BSE API error: {e}')

print()
print('=== BSE: Index Data ===')
try:
    # BSE Sensex
    sensex = yf.Ticker('^BSESN').history(period='max')
    print(f'BSE SENSEX via yfinance: {len(sensex)} records')
    print(f'Date range: {sensex.index.min().date()} → {sensex.index.max().date()}')
    print(sensex[['Close']].tail(5))
except Exception as e:
    print(f'Error: {e}')

---
## 12. Additional Python Libraries for NSE/BSE

In [ ]:
# ── 12.1 nsetools ────────────────────────────────────────────────────────────
print('=== nsetools: Stock & Index ===')

try:
    from nsetools import Nse
    nse = Nse()
    print('nsetools imported')
    
    # Get Nifty 50 top gainers
    top_gainers = nse.get_top_gainers()
    if top_gainers:
        print(f'Top gainers: {len(top_gainers)} stocks')
        if len(top_gainers) > 0:
            print(json.dumps(top_gainers[0], indent=2, default=str))
    
except ImportError:
    print('nsetools not installed or import error')
except Exception as e:
    print(f'nsetools error: {e}')

In [ ]:
# ── 12.2 MF Central API (AMFI's official portal) ─────────────────────────────
print('=== MF Central: CAS / Statement ===')

print("""
MF Central (https://www.mfcentral.com/) is the official AMFI/SEBI portal for:
- Fetching Consolidated Account Statement (CAS) by PAN
- One-click portfolio view across all AMCs
- Official transaction history

API access: Requires investor authentication with OTP/PAN.
Python library: 'casparser' (https://github.com/codereverser/casparser) — parses CAS PDF/emails.

Use case in our project:
  - User uploads CAS PDF → casparser extracts all transactions
  - No need for manual CSV entry
  - Most accurate source for personal portfolio data
""")

# Install and test casparser
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'casparser', '--quiet'])
    import casparser
    print('✅ casparser installed successfully')
    print('casparser can parse:')
    print('  - CAMS CAS PDF')
    print('  - KFin (Karvy) CAS PDF')
    print('  - Email-based CAS')
    print('  Fields: scheme, ISIN, folio, units, NAV, amount, date, transaction type')
except Exception as e:
    print(f'casparser install error: {e}')

---
## 13. Computed Analytics from NAV Data

Many critical metrics aren't available from any source — we must compute them from NAV. Let's implement and test all required calculations.

In [ ]:
# ── 13.1 Benchmark data — NSE Direct API primary, yfinance fallback ────────────
import time

# df_mftool loaded in Section 2 — extract NAV Series
df_fund_nav = df_mftool.set_index('date')[['nav']].rename(columns={'nav': 'fund_nav'})

# ── Strategy: try yfinance first (brief retry), then NSE direct API ──────────
def fetch_nifty50_via_yfinance(max_wait=60):
    """Try yfinance with a single short retry for NIFTY50."""
    import yfinance as yf
    try:
        time.sleep(2)
        h = yf.Ticker('^NSEI').history(period='max')
        if h is not None and not h.empty:
            return h[['Close']].rename(columns={'Close': 'NIFTY50'})
    except Exception as e:
        print(f'  yfinance attempt failed: {e}')
    return None

def fetch_nifty50_via_nse_direct():
    """Fetch NIFTY50 from NSE direct API (no rate limits)."""
    import requests, datetime as dt_mod
    import pandas as pd
    
    nse_headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Accept': 'application/json',
        'Referer': 'https://www.nseindia.com/'
    }
    
    session = requests.Session()
    try:
        session.get('https://www.nseindia.com/', headers=nse_headers, timeout=10)
        session.get('https://www.nseindia.com/market-data/live-equity-market',
                    headers=nse_headers, timeout=10)
    except Exception:
        pass
    
    # Fetch historical index data — NSE provides it in chunks
    today = dt_mod.date.today()
    start = dt_mod.date(2015, 1, 1)  # Go back to 2015 for rich history
    
    all_rows = []
    chunk_start = start
    while chunk_start < today:
        chunk_end = min(chunk_start + dt_mod.timedelta(days=365), today)
        url = (f'https://www.nseindia.com/api/historical/indicesHistory?'
               f'indexType=NIFTY%2050'
               f'&from={chunk_start.strftime("%d-%m-%Y")}'
               f'&to={chunk_end.strftime("%d-%m-%Y")}')
        try:
            r = session.get(url, headers=nse_headers, timeout=15)
            if r.status_code == 200:
                data = r.json()
                rows = data.get('data', {}).get('indexCloseOnlineRecords', [])
                all_rows.extend(rows)
                if len(rows) > 0:
                    print(f'  NSE chunk {chunk_start}→{chunk_end}: {len(rows)} records')
        except Exception as e:
            print(f'  NSE chunk error {chunk_start}: {e}')
        chunk_start = chunk_end + dt_mod.timedelta(days=1)
        time.sleep(0.3)
    
    if all_rows:
        df = pd.DataFrame(all_rows)
        date_col = 'EOD_TIMESTAMP' if 'EOD_TIMESTAMP' in df.columns else df.columns[0]
        close_col = 'EOD_CLOSE_INDEX_VAL' if 'EOD_CLOSE_INDEX_VAL' in df.columns else df.columns[-1]
        df['date'] = pd.to_datetime(df[date_col], format='%d-%b-%Y', errors='coerce')
        df['NIFTY50'] = pd.to_numeric(df[close_col], errors='coerce')
        df = df[['date', 'NIFTY50']].dropna().set_index('date').sort_index()
        return df
    return None

# ── Attempt fetch ────────────────────────────────────────────────────────────
bm_df = None
print('Fetching NIFTY50 benchmark data...')

# First try yfinance (fastest if not rate-limited)
print('  Trying yfinance...')
bm_yf = fetch_nifty50_via_yfinance()
if bm_yf is not None and len(bm_yf) > 100:
    bm_df = bm_yf.copy()
    bm_df.index = pd.to_datetime(bm_df.index).tz_localize(None)
    print(f'  ✓ yfinance succeeded: {len(bm_df)} records')
else:
    print('  yfinance rate-limited or no data — trying NSE direct API...')
    bm_nse = fetch_nifty50_via_nse_direct()
    if bm_nse is not None and len(bm_nse) > 100:
        bm_df = bm_nse
        print(f'  ✓ NSE direct API succeeded: {len(bm_df)} records')
    else:
        print('  ✗ NSE API also failed')

# ── Align fund NAV with benchmark ────────────────────────────────────────────
df_aligned = None
nav = None
bm  = None

if bm_df is not None:
    df_aligned = df_fund_nav.join(bm_df, how='inner').dropna()
    print(f'\n✓ Aligned records (fund + NIFTY50): {len(df_aligned)}')
    print(f'Date range: {df_aligned.index.min().date()} → {df_aligned.index.max().date()}')
    print(df_aligned.tail(5))
    
    # Define nav and bm as Series for use in downstream cells
    nav = df_aligned['fund_nav']
    bm  = df_aligned['NIFTY50']
else:
    print('\n✗ Could not get benchmark data — analytics cells will use fund-only analysis')
    # Use fund NAV alone so downstream cells don't crash
    nav = df_fund_nav['fund_nav'].dropna()
    bm  = None
    print(f'Fund NAV available: {len(nav)} records ({nav.index[0].date()} → {nav.index[-1].date()})')


In [ ]:
# ── 13.2 Core return calculations ─────────────────────────────────────────────
print('=== Analytics: Return Calculations ===')

def cagr(start_val, end_val, years):
    # Compound Annual Growth Rate
    if years <= 0 or start_val <= 0:
        return np.nan
    return ((end_val / start_val) ** (1 / years) - 1) * 100

def trailing_return(series, years):
    # Return for trailing N years; fractional years use timedelta(days)
    cutoff = series.index[-1] - pd.Timedelta(days=int(years * 365.25))
    sub = series[series.index >= cutoff]
    if len(sub) < 2:
        return np.nan
    if years >= 1:
        return cagr(sub.iloc[0], sub.iloc[-1], years)
    else:
        return (sub.iloc[-1] / sub.iloc[0] - 1) * 100

# Use nav Series (always defined from cell 62)
periods = [
    ('1M',  1/12),  ('3M',  3/12),  ('6M',  6/12),
    ('1Y',  1),     ('2Y',  2),      ('3Y',  3),
    ('5Y',  5),     ('7Y',  7),      ('10Y', 10),
]

rows = []
for label, yrs in periods:
    fund_ret = trailing_return(nav, yrs)
    bm_ret   = trailing_return(bm, yrs) if bm is not None else np.nan
    excess   = (fund_ret - bm_ret) if (not np.isnan(fund_ret) and not np.isnan(bm_ret)) else np.nan
    rows.append({
        'Period': label,
        'Fund CAGR (%)': f'{fund_ret:.2f}' if not np.isnan(fund_ret) else 'N/A',
        'NIFTY50 CAGR (%)': f'{bm_ret:.2f}' if not np.isnan(bm_ret) else 'N/A',
        'Excess Return (%)': f'{excess:+.2f}' if not np.isnan(excess) else 'N/A',
    })

print(tabulate(rows, headers='keys', tablefmt='github'))

# Since-inception return
si_start = nav.index[0]
si_yrs   = (nav.index[-1] - si_start).days / 365.25
si_ret   = cagr(nav.iloc[0], nav.iloc[-1], si_yrs)
print(f'\nSince Inception ({si_start.date()}): {si_ret:.2f}% CAGR over {si_yrs:.1f} years')

if bm is None:
    print('\nNote: Benchmark data unavailable. Run Section 8.5 separately.')


In [ ]:
# ── 13.3 Calendar year returns ────────────────────────────────────────────────
print('=== Analytics: Calendar Year Returns ===')

# Resample to last-day-of-year NAV
fund_annual = nav.resample('YE').last()
rows = []
for yr in fund_annual.index[1:]:
    prev_yr = fund_annual.index[fund_annual.index < yr]
    if len(prev_yr) == 0:
        continue
    prev_nav = fund_annual.loc[prev_yr[-1]]
    curr_nav = fund_annual.loc[yr]
    fund_ret = (curr_nav / prev_nav - 1) * 100
    
    bm_ret = np.nan
    if bm is not None:
        bm_annual = bm.resample('YE').last()
        if yr in bm_annual.index:
            prev_bm_idx = bm_annual.index[bm_annual.index < yr]
            if len(prev_bm_idx) > 0:
                bm_ret = (bm_annual.loc[yr] / bm_annual.loc[prev_bm_idx[-1]] - 1) * 100
    
    rows.append({
        'Year': yr.year,
        'Fund Return (%)': f'{fund_ret:+.2f}',
        'NIFTY50 (%)': f'{bm_ret:+.2f}' if not np.isnan(bm_ret) else 'N/A',
        'Outperformance (%)': f'{(fund_ret - bm_ret):+.2f}' if not np.isnan(bm_ret) else 'N/A',
    })

print(tabulate(rows, headers='keys', tablefmt='github'))

# Bar chart
years_  = [r['Year'] for r in rows]
fund_r  = [float(r['Fund Return (%)']) for r in rows]
fig, ax = plt.subplots(figsize=(12, 5))
colors  = ['steelblue' if v >= 0 else 'coral' for v in fund_r]
ax.bar(years_, fund_r, color=colors, alpha=0.85, label='Fund')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Calendar Year Returns — Quant Small Cap Direct Growth')
ax.set_xlabel('Year')
ax.set_ylabel('Return (%)')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── 13.4 Rolling returns ──────────────────────────────────────────────────────
print('=== Analytics: Rolling Returns ===')

def rolling_cagr(series, window_days):
    # Rolling CAGR for a given window in trading days
    yrs = window_days / 252
    return series.pct_change(window_days).apply(
        lambda x: ((1 + x) ** (1 / yrs) - 1) * 100 if not np.isnan(x) else np.nan
    )

windows = {'1Y': 252, '3Y': 756, '5Y': 1260}

for label, w in windows.items():
    roll = rolling_cagr(nav, w).dropna()
    if len(roll) < 10:
        print(f'{label}: Not enough data (need {w} trading days, have {len(nav)})')
        continue
    
    stats = [
        ('Period',          label),
        ('Min (%)',         f'{roll.min():.2f}'),
        ('Mean (%)',        f'{roll.mean():.2f}'),
        ('Max (%)',         f'{roll.max():.2f}'),
        ('Std Dev',         f'{roll.std():.2f}'),
        ('Win Rate (>0%)',  f'{(roll > 0).mean()*100:.1f}%'),
        ('Win Rate (>12%)', f'{(roll > 12).mean()*100:.1f}%'),
    ]
    print(tabulate(stats, tablefmt='github'))
    print()

# Rolling return chart for 3Y
roll_3y = rolling_cagr(nav, 756).dropna()
if len(roll_3y) >= 10:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(roll_3y.index, roll_3y, color='steelblue', linewidth=1, label='3Y Rolling CAGR')
    ax.axhline(0,  color='red',   linewidth=0.8, linestyle='--', label='0% line')
    ax.axhline(12, color='green', linewidth=0.8, linestyle='--', label='12% threshold')
    ax.fill_between(roll_3y.index, roll_3y, 0,
                    where=(roll_3y > 0), alpha=0.3, color='steelblue', label='Positive')
    ax.fill_between(roll_3y.index, roll_3y, 0,
                    where=(roll_3y < 0), alpha=0.3, color='coral',     label='Negative')
    ax.set_title('3-Year Rolling CAGR — Quant Small Cap')
    ax.set_xlabel('Date')
    ax.set_ylabel('CAGR (%)')
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# ── 13.5 Risk metrics computation ─────────────────────────────────────────────
print('=== Analytics: Risk Metrics from NAV ===')
from scipy import stats

daily_ret  = nav.pct_change().dropna()
annual_factor = 252  # trading days
rf_annual = 0.065   # 6.5% risk-free rate (approximate Indian T-bill rate)
rf_daily  = rf_annual / annual_factor

# Standard deviation (annualised)
std_ann = daily_ret.std() * np.sqrt(annual_factor) * 100

# Sharpe ratio
excess_daily = daily_ret - rf_daily
sharpe = (excess_daily.mean() / excess_daily.std()) * np.sqrt(annual_factor)

# Sortino ratio (downside deviation only)
downside = daily_ret[daily_ret < rf_daily]
sortino_denom = downside.std() * np.sqrt(annual_factor)
sortino = (excess_daily.mean() * annual_factor / sortino_denom) if sortino_denom > 0 else np.nan

# Max drawdown
running_max  = nav.cummax()
drawdown     = (nav - running_max) / running_max * 100
max_drawdown = drawdown.min()

# Beta, Alpha, R-squared (vs benchmark)
beta = alpha = r_sq = upside_cap = downside_cap = tracking_err = info_ratio = np.nan

if bm is not None:
    bm_daily = bm.pct_change().dropna()
    # Align
    aligned_ret = pd.DataFrame({'fund': daily_ret, 'bm': bm_daily}).dropna()
    
    slope, intercept, r_value, p_val, _ = stats.linregress(
        aligned_ret['bm'], aligned_ret['fund'])
    beta  = slope
    alpha = (intercept * annual_factor) * 100  # Annualised alpha
    r_sq  = r_value ** 2 * 100
    
    # Upside/downside capture
    up_periods   = aligned_ret[aligned_ret['bm'] > 0]
    down_periods = aligned_ret[aligned_ret['bm'] < 0]
    upside_cap   = (up_periods['fund'].mean()   / up_periods['bm'].mean()   * 100) if len(up_periods)   > 0 else np.nan
    downside_cap = (down_periods['fund'].mean() / down_periods['bm'].mean() * 100) if len(down_periods) > 0 else np.nan
    
    # Tracking error & information ratio
    diff_ret     = aligned_ret['fund'] - aligned_ret['bm']
    tracking_err = diff_ret.std() * np.sqrt(annual_factor) * 100
    ann_diff     = diff_ret.mean() * annual_factor * 100
    info_ratio   = ann_diff / tracking_err if tracking_err > 0 else np.nan

metrics = [
    ('Std Deviation (annualised, %)', f'{std_ann:.2f}%'),
    ('Sharpe Ratio',                  f'{sharpe:.3f}'),
    ('Sortino Ratio',                 f'{sortino:.3f}'),
    ('Max Drawdown (%)',              f'{max_drawdown:.2f}%'),
    ('Beta',                          f'{beta:.3f}' if not np.isnan(beta) else 'N/A (no bm)'),
    ('Alpha (annualised, %)',         f'{alpha:.2f}%' if not np.isnan(alpha) else 'N/A (no bm)'),
    ('R-Squared (%)',                 f'{r_sq:.2f}%' if not np.isnan(r_sq) else 'N/A (no bm)'),
    ('Upside Capture (%)',            f'{upside_cap:.1f}%' if not np.isnan(upside_cap) else 'N/A (no bm)'),
    ('Downside Capture (%)',          f'{downside_cap:.1f}%' if not np.isnan(downside_cap) else 'N/A (no bm)'),
    ('Tracking Error (%)',            f'{tracking_err:.2f}%' if not np.isnan(tracking_err) else 'N/A (no bm)'),
    ('Information Ratio',             f'{info_ratio:.3f}' if not np.isnan(info_ratio) else 'N/A (no bm)'),
    ('Risk-Free Rate Used (%)',       f'{rf_annual*100:.1f}%'),
]
print(tabulate(metrics, headers=['Metric', 'Value'], tablefmt='github'))

if bm is None:
    print('\nNote: Beta, Alpha, R2, Capture Ratios need benchmark data.')
    print('      Fetch benchmark separately (Section 8.4/8.5) and rerun this cell.')


In [ ]:
# ── 13.6 SIP simulation ───────────────────────────────────────────────────────
print('=== Analytics: SIP Simulation ===')
from scipy.optimize import brentq

def simulate_sip(nav_series, monthly_amount=10000, start_date=None):
    # Simulate monthly SIP and compute XIRR
    if start_date is None:
        start_date = nav_series.index[0]

    nav_series = nav_series.copy()
    nav_series.index = pd.to_datetime(nav_series.index)

    monthly = nav_series.resample('MS').first().dropna()
    monthly = monthly[monthly.index >= pd.Timestamp(start_date)]

    if len(monthly) == 0:
        return None

    units_held = 0
    invested   = 0
    cashflows  = []
    dates      = []

    for date_, nav_val in monthly.items():
        units_bought = monthly_amount / nav_val
        units_held  += units_bought
        invested    += monthly_amount
        cashflows.append(-monthly_amount)
        dates.append(date_)

    final_nav   = nav_series.iloc[-1]
    final_value = units_held * final_nav
    cashflows.append(final_value)
    dates.append(nav_series.index[-1])

    def xnpv(rate, cashflows, dates):
        t0 = dates[0]
        return sum(cf / (1 + rate) ** ((d - t0).days / 365)
                   for cf, d in zip(cashflows, dates))

    try:
        xirr = brentq(lambda r: xnpv(r, cashflows, dates), -0.5, 100)
    except Exception:
        xirr = np.nan

    return {
        'Total Invested (Rs)':  invested,
        'Current Value (Rs)':   final_value,
        'Absolute Gain (Rs)':   final_value - invested,
        'Absolute Return (%)':  (final_value / invested - 1) * 100,
        'XIRR (%)':             xirr * 100 if not np.isnan(xirr) else np.nan,
        'Units Held':           units_held,
        'SIP Instalments':      len(monthly),
        'Avg Cost (Rs/unit)':   invested / units_held,
        'Current NAV (Rs)':     final_nav,
    }

# nav is always defined from Cell 62
five_yr_start   = nav.index[-1] - pd.DateOffset(years=5)
three_yr_start  = nav.index[-1] - pd.DateOffset(years=3)
inception_start = nav.index[0]

for label, start in [('5-Year SIP',         five_yr_start),
                     ('3-Year SIP',         three_yr_start),
                     ('Since-Inception SIP', inception_start)]:
    result = simulate_sip(nav, monthly_amount=10000, start_date=start)
    if result:
        print(f'\n--- {label} (Rs 10,000/month) ---')
        for k, v in result.items():
            if isinstance(v, float):
                print(f'  {k:25s}: {v:,.2f}')
            else:
                print(f'  {k:25s}: {v}')


In [ ]:
# ── 13.7 Drawdown chart ───────────────────────────────────────────────────────
print('=== Analytics: Drawdown Chart ===')

nav_norm    = nav / nav.iloc[0] * 100
running_max_nav = nav_norm.cummax()
drawdown_series = (nav_norm - running_max_nav) / running_max_nav * 100

if bm is not None:
    bm_norm     = bm / bm.iloc[0] * 100
    running_max_bm = bm_norm.cummax()
    dd_bm = (bm_norm - running_max_bm) / running_max_bm * 100
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    ax1.plot(nav_norm.index, nav_norm, label='Fund (indexed ₹100)', color='steelblue', linewidth=1.2)
    ax1.plot(bm_norm.index,  bm_norm,  label='NIFTY50 (indexed ₹100)', color='coral', linestyle='--', linewidth=1)
    ax1.set_ylabel('Value (₹100 base)')
    ax1.legend()
    ax1.set_title('Fund vs NIFTY50 — Indexed Performance')
    
    ax2.fill_between(drawdown_series.index, drawdown_series, 0,
                     label='Fund Drawdown', color='steelblue', alpha=0.4)
    ax2.fill_between(dd_bm.index, dd_bm, 0,
                     label='NIFTY50 Drawdown', color='coral', alpha=0.3)
    ax2.set_ylabel('Drawdown from Peak (%)')
    ax2.set_xlabel('Date')
    ax2.legend()
    ax2.set_title('Drawdown from All-Time Peak')
    
    plt.tight_layout()
    plt.show()
    
    print(f'Fund Max Drawdown   : {drawdown_series.min():.2f}%')
    print(f'NIFTY50 Max Drawdown: {dd_bm.min():.2f}%')
else:
    # Single panel — fund only
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    ax1.plot(nav_norm.index, nav_norm, color='steelblue', linewidth=1.2, label='Fund NAV (indexed)')
    ax1.set_ylabel('Value (₹100 base)')
    ax1.legend()
    ax1.set_title('Fund NAV — Indexed Performance (benchmark unavailable)')
    
    ax2.fill_between(drawdown_series.index, drawdown_series, 0,
                     color='steelblue', alpha=0.4, label='Fund Drawdown')
    ax2.set_ylabel('Drawdown from Peak (%)')
    ax2.set_xlabel('Date')
    ax2.legend()
    ax2.set_title('Drawdown from All-Time Peak')
    
    plt.tight_layout()
    plt.show()
    
    print(f'Fund Max Drawdown: {drawdown_series.min():.2f}%')
    print('Benchmark chart unavailable — fetch NIFTY50 data in Section 8.5 and rerun.')


---
## 14. Forecasting Approaches

From `docs/data-source-exploration.md` resource #4: https://github.com/NayakwadiS/Forecasting_Mutual_Funds  
Testing NAV forecasting methods — **for research only, not investment advice**.

In [ ]:
# ── 14.1 Simple Moving Average / Trend Analysis ───────────────────────────────
print('=== Trend Analysis: Moving Averages (illustrative — not predictive) ===')
print('⚠️  These are for trend visualisation only — past patterns do NOT predict returns')

# nav is always defined from Cell 62
df_forecast = nav.copy().iloc[-504:]  # ~2 years of trading days

sma20  = df_forecast.rolling(20).mean()
sma50  = df_forecast.rolling(50).mean()
sma200 = df_forecast.rolling(200).mean()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df_forecast.index, df_forecast, label='NAV', color='steelblue', linewidth=1.2)
ax.plot(sma20.index,  sma20,  label='20-day SMA',  color='orange', linestyle='--', linewidth=1)
ax.plot(sma50.index,  sma50,  label='50-day SMA',  color='red',    linestyle='--', linewidth=1)
ax.plot(sma200.index, sma200, label='200-day SMA', color='purple', linestyle=':',  linewidth=1)
ax.set_title('NAV with Moving Averages (trend analysis — not a trading signal)')
ax.set_xlabel('Date')
ax.set_ylabel('NAV (₹)')
ax.legend()
plt.tight_layout()
plt.show()

# Current trend status
last_nav   = df_forecast.iloc[-1]
last_sma20 = sma20.iloc[-1]
last_sma50 = sma50.iloc[-1]
print(f'\nCurrent NAV  : ₹{last_nav:.4f}')
print(f'20-day SMA   : ₹{last_sma20:.4f}  → NAV {"above" if last_nav > last_sma20 else "below"} 20-day SMA')
print(f'50-day SMA   : ₹{last_sma50:.4f}  → NAV {"above" if last_nav > last_sma50 else "below"} 50-day SMA')


In [ ]:
# ── 14.2 Momentum signals (for backtesting research) ─────────────────────────
print('=== Backtesting Signals: Momentum & MA Filter ===')
print('ℹ️  These signals are from the workflow.md backtesting strategy research')

# nav is always defined from Cell 62
momentum_12m = nav / nav.shift(252) - 1
signal_momentum = (momentum_12m > 0).astype(int)

ma_10m      = nav.rolling(210).mean()  # ~10 months of trading days
signal_ma   = (nav > ma_10m).astype(int)

vol_6m      = nav.pct_change().rolling(126).std() * np.sqrt(252)
vol_threshold = vol_6m.quantile(0.7)
signal_vol  = (vol_6m < vol_threshold).astype(int)

combined_signal = (signal_momentum + signal_ma) / 2

# Stats (last 3 years)
cutoff = nav.index[-1] - pd.DateOffset(years=3)
last3yr = nav.index >= cutoff

stats_rows = [
    ('12M Momentum "IN equity"',       f'{signal_momentum[last3yr].mean()*100:.1f}%'),
    ('10M MA filter "IN equity"',      f'{signal_ma[last3yr].mean()*100:.1f}%'),
    ('Low volatility "IN equity"',     f'{signal_vol[last3yr].mean()*100:.1f}%'),
    ('Combined (momentum+MA) "IN"',    f'{combined_signal[last3yr].mean()*100:.1f}%'),
]
print('\nSignal Statistics (last 3 years):')
print(tabulate(stats_rows, headers=['Signal', '% Days Invested'], tablefmt='github'))

# Plot combined signal
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
ax1.plot(nav.index, nav, color='steelblue', linewidth=1, label='Fund NAV')
ax1.set_ylabel('NAV (₹)')
ax1.legend()
ax1.set_title('Fund NAV with Combined Momentum+MA Signal')

ax2.fill_between(combined_signal.index, combined_signal, 0, 
                 alpha=0.6, color='green', label='IN equity (signal=1)')
ax2.fill_between(combined_signal.index, 1 - combined_signal, 0, 
                 alpha=0.3, color='orange', label='OUT (debt/cash)')
ax2.set_ylabel('Signal Strength')
ax2.set_xlabel('Date')
ax2.legend()
ax2.set_title('Combined Momentum + Moving Average Signal')
plt.tight_layout()
plt.show()


---
## 15. News & Sentiment (Research)

From resource: https://amaltyagi.medium.com/fetching-news-sentiment-with-python-5c2a0888e681

In [ ]:
# ── 15.1 NewsAPI / RSS feeds ──────────────────────────────────────────────────
print('=== News Sources for Portfolio Context ===')

# Test RSS feeds for mutual fund news (no auth required)
news_feeds = [
    ('MoneyControl MF',    'https://www.moneycontrol.com/rss/MFnews.xml'),
    ('Economic Times MF',  'https://economictimes.indiatimes.com/mf/rss'),
    ('ValueResearch',      'https://www.valueresearchonline.com/feeds/news.aspx'),
    ('AMFI Press',         'https://www.amfiindia.com/news-and-press-releases'),
]

print('Documented news RSS/API sources:')
print(tabulate(news_feeds, headers=['Source','URL'], tablefmt='github'))
print()

# Try fetching one feed
try:
    import feedparser
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'feedparser', '--quiet'])
    import feedparser

try:
    feed = feedparser.parse('https://www.moneycontrol.com/rss/MFnews.xml')
    print(f'MoneyControl MF feed: {len(feed.entries)} articles')
    for entry in feed.entries[:3]:
        print(f"  📰 {entry.get('title','N/A')[:80]}")
        print(f"     {entry.get('published','N/A')}")
except Exception as e:
    print(f'Feed error: {e}')

---
## 16. Data Coverage Matrix & Recommendations

Summary of what each source provides and the recommended strategy.

In [ ]:
# ── 16.1 Master data coverage matrix ─────────────────────────────────────────

coverage = {
    'Data Need': [
        'Scheme Universe (all Indian MFs)',
        'Current NAV',
        'Historical NAV (full since inception)',
        'Scheme type / category',
        'Expense ratio',
        'AUM',
        'Exit load details',
        'Min investment (lumpsum & SIP)',
        'SIP / STP / SWP availability',
        'Lock-in period',
        'Fund manager name',
        'Fund manager tenure',
        'Fund manager history',
        'Benchmark index mapping',
        'Portfolio holdings (equity)',
        'Portfolio holdings (debt)',
        'Sector allocation',
        'Market cap allocation (large/mid/small)',
        'Asset allocation (eq/debt/cash %)',
        'Portfolio turnover',
        'Morningstar star ratings',
        'Trailing returns (pre-computed)',
        'Category avg returns for comparison',
        'Rank within category',
        'Std Dev / Beta / Alpha / Sharpe / Sortino',
        'Max Drawdown (pre-computed)',
        'Upside/Downside Capture Ratio',
        'P/E Ratio of portfolio',
        'CRISIL rating',
        'Benchmark index history',
        'Transaction-level XIRR (user data)',
        'CAS PDF parsing (portfolio import)',
        'News / sentiment',
    ],
    'mftool': [
        '✅','✅','✅','✅','❌','❌','❌','❌','❌','❌',
        '❌','❌','❌','❌','❌','❌','❌','❌','❌','❌',
        '❌','❌','❌','❌','❌','❌','❌','❌','❌','✅*',
        '❌','❌','❌',
    ],
    'mfapi.in': [
        '✅','✅','✅','✅','❌','❌','❌','❌','❌','❌',
        '❌','❌','❌','❌','❌','❌','❌','❌','❌','❌',
        '❌','❌','❌','❌','❌','❌','❌','❌','❌','✅*',
        '❌','❌','❌',
    ],
    'captnemo': [
        '⚠️','✅','⚠️','✅','✅','✅','⚠️','✅','✅','✅',
        '⚠️','❌','❌','❌','❌','❌','❌','❌','❌','❌',
        '❌','✅','❌','❌','⚠️','❌','❌','❌','✅','✅*',
        '❌','❌','❌',
    ],
    'yfinance': [
        '❌','✅','✅','⚠️','⚠️','⚠️','❌','❌','❌','❌',
        '❌','❌','❌','❌','❌','❌','❌','❌','❌','❌',
        '⚠️','⚠️','❌','❌','⚠️','❌','❌','❌','❌','✅',
        '❌','❌','❌',
    ],
    'yahooquery': [
        '❌','✅','✅','⚠️','⚠️','⚠️','❌','❌','❌','❌',
        '❌','❌','❌','❌','✅','⚠️','✅','✅','✅','❌',
        '⚠️','✅','❌','❌','⚠️','❌','❌','⚠️','❌','✅',
        '❌','❌','❌',
    ],
    'mstarpy': [
        '⚠️','✅','✅','✅','⚠️','⚠️','❌','❌','❌','❌',
        '⚠️','❌','❌','❌','✅','✅','✅','✅','✅','❌',
        '✅','✅','✅','✅','✅','✅','❌','⚠️','❌','✅*',
        '❌','❌','❌',
    ],
    'Computed': [
        '❌','❌','❌','❌','❌','❌','❌','❌','❌','❌',
        '❌','❌','❌','❌','❌','❌','❌','❌','❌','❌',
        '❌','✅','✅','❌','✅','✅','✅','❌','❌','❌',
        '✅','❌','❌',
    ],
    'casparser': [
        '❌','❌','❌','❌','❌','❌','❌','❌','❌','❌',
        '❌','❌','❌','❌','❌','❌','❌','❌','❌','❌',
        '❌','❌','❌','❌','❌','❌','❌','❌','❌','❌',
        '❌','✅','❌',
    ],
}

df_coverage = pd.DataFrame(coverage).set_index('Data Need')
print('=== Complete Data Coverage Matrix ===')
print(df_coverage.to_string())
print()
print('Legend: ✅ = Available  ⚠️ = Partial/Unreliable  ❌ = Not Available  * = via yfinance')

---
## 17. Recommended Architecture

Based on the exploration above, here is the recommended data strategy for the backend.

In [ ]:
# ── 17.1 Recommended data layer strategy ─────────────────────────────────────

architecture = [
    ('1. Scheme Universe & NAV History',
     'PRIMARY: mftool + mfapi.in (both read AMFI NAVAll.txt)',
     'Fetch all ~4000 Direct Growth schemes; daily NAV update'),
    
    ('2. Enrichment Metadata\n   (expense, loads, AUM, SIP)',
     'PRIMARY: mf.captnemo.in (Kuvera proxy)',
     'Schedule weekly refresh; handle missing fields gracefully'),
    
    ('3. Risk Metrics\n   (Std Dev, Sharpe, Beta, Alpha, MDD)',
     'PRIMARY: Compute from NAV vs benchmark\n   SECONDARY: mstarpy for cross-validation',
     'Compute nightly from stored NAV data; cache results'),
    
    ('4. Portfolio Holdings & Sectors',
     'PRIMARY: mstarpy (most complete)\n   SECONDARY: yahooquery (top 10 only)',
     'Monthly refresh aligns with AMC disclosure cycle'),
    
    ('5. Morningstar Ratings & Category Returns',
     'PRIMARY: mstarpy',
     'Include category avg and rank for screener comparison'),
    
    ('6. Benchmark Indices',
     'PRIMARY: yfinance (^NSEI, NIFMID150.NS, etc.)\n   SECONDARY: NSE direct API',
     'Map each fund category to correct benchmark; daily update'),
    
    ('7. Fund Manager Info',
     'BEST AVAILABLE: mstarpy (limited)\n   FALLBACK: Manual curation / scraping AMC sites',
     'Critical gap — consider scraping scheme info documents (SID)'),
    
    ('8. User Portfolio Import',
     'PRIMARY: casparser (CAS PDF)\n   SECONDARY: Manual CSV upload',
     'CAS parser gives complete transaction history across all AMCs'),
    
    ('9. XIRR & Portfolio Analytics',
     'COMPUTED from user transactions + fetched NAV data',
     'Core backend calculation engine'),
    
    ('10. News Context',
     'RSS feeds (MoneyControl, ET Markets)',
     'Optional enrichment for portfolio review section'),
]

print('=== Recommended Backend Data Architecture ===')
print()
for i, (need, source, notes) in enumerate(architecture, 1):
    print(f'📌 {need}')
    print(f'   Source : {source}')
    print(f'   Notes  : {notes}')
    print()

In [ ]:
# ── 17.2 Data gaps that require solutions ────────────────────────────────────
print('=== Known Data Gaps & Proposed Solutions ===')

gaps = [
    ('Fund manager detailed profile\n(tenure, background, other funds)',
     '❌ No clean API',
     'Option A: Scrape AMC websites (legal risk)\nOption B: Manual curation for top-N funds\nOption C: Parse SID PDFs using pdfplumber'),
    
    ('Historical expense ratio changes',
     '❌ No API',
     'Best approach: Snapshot expense ratio periodically and store historically'),
    
    ('Historical AUM (time series)',
     '⚠️ Sparse',
     'AMFI publishes monthly AUM data → parse PDF/HTML reports'),
    
    ('Exit load schedule details',
     '⚠️ Partial via captnemo',
     'Parse from SID documents or maintain lookup table'),
    
    ('Historical portfolio holdings',
     '❌ Not available via API',
     'AMC websites publish monthly PDFs → parse them for holdings history'),
    
    ('P/E ratio of fund portfolio',
     '⚠️ Sometimes via yahooquery/mstarpy',
     'Compute from holdings × individual stock P/E ratios'),
    
    ('Benchmark index for each fund',
     '❌ No automated mapping',
     'Create/maintain a lookup table: AMFI_code → benchmark Yahoo symbol'),
    
    ('PE-based valuation signals (backtesting)',
     '⚠️ Partial',
     'NSE publishes daily P/E data for NIFTY indices; yfinance has some'),
    
    ('Direct vs Regular plan comparison',
     '✅ Both codes in AMFI',
     'Maintain mapping: each Regular plan → corresponding Direct plan'),
]

print(tabulate(
    [(g, s, r) for g, s, r in gaps],
    headers=['Gap', 'Current Status', 'Proposed Solution'],
    tablefmt='grid'
))

In [ ]:
# ── 17.3 Recommended fund-benchmark mapping ───────────────────────────────────
print('=== Recommended Benchmark Mapping (Category → Index) ===')

benchmark_map = [
    # (SEBI Category,                    Benchmark Name,           Yahoo Symbol)
    ('Equity - Large Cap',               'NIFTY 100 TRI',          '^CNX100'),
    ('Equity - Mid Cap',                 'NIFTY Midcap 150 TRI',   'NIFMID150.NS'),
    ('Equity - Small Cap',               'NIFTY Smallcap 250 TRI', 'NIFSMCP250.NS'),
    ('Equity - Large & Mid Cap',         'NIFTY LargeMidcap 250',  'NIFLARGEMID250.NS'),
    ('Equity - Flexi Cap',               'NIFTY 500 TRI',          '^CRSLDX'),
    ('Equity - Multi Cap',               'NIFTY 500 Multicap 50:25:25', 'NIFTY500MULT.NS'),
    ('Equity - ELSS',                    'NIFTY 500 TRI',          '^CRSLDX'),
    ('Equity - Value/Contra',            'NIFTY 500 TRI',          '^CRSLDX'),
    ('Equity - Focused',                 'NIFTY 500 TRI',          '^CRSLDX'),
    ('Debt - Liquid',                    'NIFTY Liquid Index',      None),
    ('Debt - Overnight',                 'NIFTY 1D Rate Index',     None),
    ('Debt - Ultra Short',               'CRISIL Ultra Short Term', None),
    ('Debt - Short Term',                'NIFTY Short Duration Debt', None),
    ('Debt - Medium Duration',           'NIFTY Medium Duration Debt', None),
    ('Debt - Long Term',                 'NIFTY Long Duration Debt', None),
    ('Debt - Corporate Bond',            'NIFTY Corporate Bond',   None),
    ('Debt - Credit Risk',               'NIFTY Credit Risk Bond',  None),
    ('Debt - Gilt',                      'NIFTY All Duration G-Sec', None),
    ('Hybrid - Aggressive',              'NIFTY 50 + NIFTY Short Term (65:35)', '^NSEI'),
    ('Hybrid - Balanced Advantage',      'NIFTY 50 Hybrid Composite', None),
    ('International - US Equity',        'NASDAQ-100 / S&P 500',   '^NDX'),
    ('Index Fund - NIFTY 50',            'NIFTY 50 TRI',           '^NSEI'),
    ('Index Fund - NIFTY Next 50',       'NIFTY Next 50 TRI',      '^NSMIDCP'),
]

print(tabulate(benchmark_map, 
    headers=['SEBI Category', 'Benchmark Index', 'Yahoo Finance Symbol'],
    tablefmt='github'))

print()
print('Note: "TRI" = Total Return Index (includes dividends); plain NSE indices do NOT include dividends.')
print('For fair comparison, always use TRI benchmarks.')

In [ ]:
# ── 17.4 Final recommendations ────────────────────────────────────────────────

print('''
╔══════════════════════════════════════════════════════════════════════════════╗
║          FINAL DATA STRATEGY RECOMMENDATIONS                               ║
╚══════════════════════════════════════════════════════════════════════════════╝

PHASE 1 DATA FOUNDATION:
─────────────────────────
  1. NAV Data Pipeline:
     • Daily: Fetch AMFI NAVAll.txt → parse → store in DB
     • Source: mftool or mfapi.in (both reliable, mfapi.in is REST-first)
     • Coverage: ALL ~4000 Direct Growth plans

  2. Enrichment (run weekly):
     • mf.captnemo.in → expense ratio, AUM, min investment, loads
     • mstarpy → Morningstar ratings, trailing returns, category avg
     • yahooquery → top holdings, sector weights

  3. Benchmark Indices (daily via yfinance):
     • Store NIFTY 50, Midcap 150, Smallcap 250, NIFTY 500
     • Map each AMFI code → benchmark in a lookup table

  4. Compute nightly (from stored NAV + benchmark data):
     • Trailing returns (1M, 3M, 6M, 1Y, 2Y, 3Y, 5Y, 10Y, SI)
     • Calendar year returns, rolling returns (1Y, 3Y, 5Y)
     • Std Dev, Beta, Alpha, Sharpe, Sortino, R², Max Drawdown
     • Upside/Downside Capture, Information Ratio, Tracking Error

PHASE 2 PORTFOLIO ANALYSIS:
─────────────────────────────
  5. casparser → Parse CAS PDF → extract all user transactions
  6. XIRR calculation (scipy.optimize.brentq on cashflows)
  7. Benchmark simulation (replay investor cashflows on benchmark NAV)
  8. Holdings from mstarpy → portfolio overlap detection

KEY GAPS (Workarounds):
───────────────────────
  • Fund manager details  → Manual curation / SID PDF parsing
  • Historical AUM        → Monthly AMFI PDF parsing
  • P/E of portfolio      → Compute from holdings + NSE P/E data
  • TRI benchmarks        → Cross-validate yfinance with NSE direct
  • Historical expense ratio → Snapshot on each weekly fetch, store timeseries

LICENSING & RELIABILITY:
─────────────────────────
  • AMFI data  → Public, no restrictions
  • mstarpy    → Morningstar public data, check ToS before production use
  • captnemo   → Kuvera proxy, may change without notice
  • yfinance   → Yahoo unofficial API, use responsibly
  • NSE APIs   → Register as developer (https://www.nseindia.com/)
''')

print('✅ Exploration complete. Ready to begin Phase 1 backend implementation.')

---

## Summary

| Source | Primary Role | Reliability | Key Limitation |
|--------|-------------|-------------|----------------|
| **mftool / mfapi.in** | NAV history, scheme universe | ⭐⭐⭐⭐⭐ | No enrichment data |
| **mf.captnemo.in** | Expense ratio, AUM, loads, min investment | ⭐⭐⭐⭐ | Depends on Kuvera |
| **mstarpy** | Risk metrics, ratings, holdings, sector | ⭐⭐⭐⭐⭐ | Expense ratio unavailable for India |
| **yfinance** | Benchmark indices (NIFTY, SENSEX) | ⭐⭐⭐⭐ | Ticker coverage gaps for MFs |
| **yahooquery** | Holdings, sector weights (top 10) | ⭐⭐⭐ | Inconsistent for Indian MFs |
| **AMFI Direct** | Authoritative NAV source | ⭐⭐⭐⭐⭐ | Raw text format needs parsing |
| **casparser** | CAS PDF → transaction import | ⭐⭐⭐⭐⭐ | Requires user to upload PDF |
| **Computed** | Risk metrics, rolling returns, XIRR | ⭐⭐⭐⭐⭐ | Requires clean input data |

> **Next step:** Build the Phase 1 data ingestion pipeline using mftool/mfapi.in + mf.captnemo.in + mstarpy, with nightly computation of all derived analytics.

---
*Disclaimer: All fund data shown in this notebook is for educational and research purposes only. It does not constitute financial advice.*